# Moondream2 Gaze Evaluation

In [2]:
import json, os
from datetime import datetime, timezone
from pathlib import Path
from datasets import load_dataset
from transformers import AutoModelForCausalLM
import torch
from tqdm.auto import tqdm
import numpy as np
import pandas as pd


RESULTS_PATH = Path("./MoondreamEval.json")
REVISION = "2025-01-09"
SAVE_INTERVAL = 50

# Evaluation

In [ ]:
dataset = load_dataset("Zory/gaze-referent-stimuli", split="train")
print(f"Total stimuli in dataset: {len(dataset)}")

Generating train split: 100%|██████████| 2273/2273 [00:00<00:00, 6519.04 examples/s]


Total stimuli in dataset: 2273


In [ ]:
existing_results = json.loads(RESULTS_PATH.read_text()) if RESULTS_PATH.exists() else {}
evaluated_ids = set(existing_results.keys())
to_evaluate = [entry for entry in dataset if entry["stimulus_id"] not in evaluated_ids]
print(f"Already evaluated: {len(evaluated_ids)}, Remaining: {len(to_evaluate)}")

model = AutoModelForCausalLM.from_pretrained(
    "vikhyatk/moondream2",
    trust_remote_code=True,
    dtype=torch.float16,
    device_map={"": "mps"},
    revision=REVISION,
    low_cpu_mem_usage=True
)
# conda install -c conda-forge libvips

for i, entry in enumerate(tqdm(to_evaluate, desc="Evaluating")):
    face = {
        "x_min": entry["face_x_min"],
        "y_min": entry["face_y_min"],
        "x_max": entry["face_x_max"],
        "y_max": entry["face_y_max"]
    }
    
    gaze = model.detect_gaze(
        entry["image"],
        eye=None,
        face=face,
        unstable_settings={"prioritize_accuracy": True, "force_detect": True}
    )["gaze"]
    
    existing_results[entry["stimulus_id"]] = {
        "gaze": {"x": gaze["x"], "y": gaze["y"]} if gaze else None,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC"),
        "revision": REVISION
    }
    
    if (i + 1) % SAVE_INTERVAL == 0:
        RESULTS_PATH.write_text(json.dumps(existing_results, indent=2))
    
RESULTS_PATH.write_text(json.dumps(existing_results, indent=2))

# Analysis

In [15]:
results = json.loads(RESULTS_PATH.read_text())

def get_condition(entry):
    """Determine condition: Stimuli_A, Natural, Consistent, Inconsistent"""
    if entry["site"] == "Site0":
        return "Stimuli_A"
    else:
        head_target = entry["head_target"]
        if head_target == "-1":
            return "Natural"
        elif head_target == entry["label"]:
            return "Consistent"
        else:
            return "Inconsistent"

def get_options(entry):
    """Get all valid option positions and their names"""
    candidates = entry["candidates"].split("+")
    options = []
    for i, name in enumerate(candidates, start=1):
        x = entry[f"option{i}_x"]
        y = entry[f"option{i}_y"]
        if x >= 0 and y >= 0:
            options.append({"name": name, "x": x, "y": y})
    return options

def get_ground_truth(entry):
    """Get ground truth position based on label"""
    candidates = entry["candidates"].split("+")
    label = entry["label"]
    for i, name in enumerate(candidates, start=1):
        if name == label:
            x = entry[f"option{i}_x"]
            y = entry[f"option{i}_y"]
            if x >= 0 and y >= 0:
                return {"name": name, "x": x, "y": y}
    return None

# Collect metrics by condition
conditions = ["Stimuli_A", "Natural", "Consistent", "Inconsistent"]
metrics_by_condition = {c: {"L2": [], "DX": [], "Acc_L2": [], "Acc_DX": []} for c in conditions}

missing_gaze = 0

for entry in dataset:
    stim_id = str(entry["stimulus_id"])
    if stim_id not in results:
        continue
    
    result = results[stim_id]
    if result["gaze"] is None:
        missing_gaze += 1
        continue
    
    pred_x, pred_y = result["gaze"]["x"], result["gaze"]["y"]
    
    gt = get_ground_truth(entry)
    if gt is None:
        continue
    
    options = get_options(entry)
    if len(options) < 2:
        continue
    
    condition = get_condition(entry)
    
    # L2 distance to ground truth
    l2_dist = np.sqrt((pred_x - gt["x"])**2 + (pred_y - gt["y"])**2)
    metrics_by_condition[condition]["L2"].append(l2_dist)
    
    # DX (x-axis) distance to ground truth
    dx_dist = np.abs(pred_x - gt["x"])
    metrics_by_condition[condition]["DX"].append(dx_dist)
    
    # Accuracy using L2 (nearest option)
    nearest_l2 = min(options, key=lambda o: np.sqrt((pred_x - o["x"])**2 + (pred_y - o["y"])**2))
    acc_l2 = 1 if nearest_l2["name"] == gt["name"] else 0
    metrics_by_condition[condition]["Acc_L2"].append(acc_l2)
    
    # Accuracy using DX (nearest in x-axis)
    nearest_dx = min(options, key=lambda o: np.abs(pred_x - o["x"]))
    acc_dx = 1 if nearest_dx["name"] == gt["name"] else 0
    metrics_by_condition[condition]["Acc_DX"].append(acc_dx)

print(f"Total evaluated: {len(results)}")
print(f"Missing gaze predictions: {missing_gaze}\n")

# Build performance grid
grid_data = []
for condition in conditions:
    m = metrics_by_condition[condition]
    if len(m["L2"]) == 0:
        continue
    grid_data.append({
        "Condition": condition,
        "N": len(m["L2"]),
        "L2": f"{np.mean(m['L2']):.4f}",
        "DX": f"{np.mean(m['DX']):.4f}",
        "Acc_L2": f"{np.mean(m['Acc_L2'])*100:.1f}%",
        "Acc_DX": f"{np.mean(m['Acc_DX'])*100:.1f}%"
    })

# Add aggregated Stimuli B row
stimuli_b_conditions = ["Natural", "Consistent", "Inconsistent"]
all_l2, all_dx, all_acc_l2, all_acc_dx = [], [], [], []
for c in stimuli_b_conditions:
    m = metrics_by_condition[c]
    all_l2.extend(m["L2"])
    all_dx.extend(m["DX"])
    all_acc_l2.extend(m["Acc_L2"])
    all_acc_dx.extend(m["Acc_DX"])

if len(all_l2) > 0:
    grid_data.append({
        "Condition": "Stimuli_B (agg)",
        "N": len(all_l2),
        "L2": f"{np.mean(all_l2):.4f}",
        "DX": f"{np.mean(all_dx):.4f}",
        "Acc_L2": f"{np.mean(all_acc_l2)*100:.1f}%",
        "Acc_DX": f"{np.mean(all_acc_dx)*100:.1f}%"
    })

df = pd.DataFrame(grid_data)
print("Performance by Condition (5 x 4 grid):\n")
print(df.to_string(index=False))

Total evaluated: 2273
Missing gaze predictions: 0

Performance by Condition (5 x 4 grid):

      Condition    N     L2     DX Acc_L2 Acc_DX
      Stimuli_A  907 0.1608 0.1198  54.8%  53.6%
        Natural  333 0.1705 0.1324  57.1%  56.2%
     Consistent  332 0.1602 0.1194  60.2%  59.6%
   Inconsistent  701 0.3221 0.2916  23.5%  23.5%
Stimuli_B (agg) 1366 0.2458 0.2109  40.6%  40.3%


In [16]:
# Further breakdown by angle (left, front, right)
angles = ["left", "front", "right"]

# Collect metrics by condition x angle
metrics_by_cond_angle = {}
for c in conditions:
    for a in angles:
        metrics_by_cond_angle[(c, a)] = {"L2": [], "DX": [], "Acc_L2": [], "Acc_DX": []}

for entry in dataset:
    stim_id = str(entry["stimulus_id"])
    if stim_id not in results:
        continue
    
    result = results[stim_id]
    if result["gaze"] is None:
        continue
    
    pred_x, pred_y = result["gaze"]["x"], result["gaze"]["y"]
    
    gt = get_ground_truth(entry)
    if gt is None:
        continue
    
    options = get_options(entry)
    if len(options) < 2:
        continue
    
    condition = get_condition(entry)
    angle = entry["angle"]
    
    key = (condition, angle)
    if key not in metrics_by_cond_angle:
        continue
    
    # L2 distance to ground truth
    l2_dist = np.sqrt((pred_x - gt["x"])**2 + (pred_y - gt["y"])**2)
    metrics_by_cond_angle[key]["L2"].append(l2_dist)
    
    # DX (x-axis) distance to ground truth
    dx_dist = np.abs(pred_x - gt["x"])
    metrics_by_cond_angle[key]["DX"].append(dx_dist)
    
    # Accuracy using L2 (nearest option)
    nearest_l2 = min(options, key=lambda o: np.sqrt((pred_x - o["x"])**2 + (pred_y - o["y"])**2))
    acc_l2 = 1 if nearest_l2["name"] == gt["name"] else 0
    metrics_by_cond_angle[key]["Acc_L2"].append(acc_l2)
    
    # Accuracy using DX (nearest in x-axis)
    nearest_dx = min(options, key=lambda o: np.abs(pred_x - o["x"]))
    acc_dx = 1 if nearest_dx["name"] == gt["name"] else 0
    metrics_by_cond_angle[key]["Acc_DX"].append(acc_dx)

# Build performance grid: 12 rows (4 conditions x 3 angles) x 4 metrics
grid_data = []
for condition in conditions:
    for angle in angles:
        key = (condition, angle)
        m = metrics_by_cond_angle[key]
        if len(m["L2"]) == 0:
            continue
        grid_data.append({
            "Condition": condition,
            "Angle": angle,
            "N": len(m["L2"]),
            "L2": f"{np.mean(m['L2']):.4f}",
            "DX": f"{np.mean(m['DX']):.4f}",
            "Acc_L2": f"{np.mean(m['Acc_L2'])*100:.1f}%",
            "Acc_DX": f"{np.mean(m['Acc_DX'])*100:.1f}%"
        })

df_angle = pd.DataFrame(grid_data)
print("Performance by Condition x Angle (12 x 4 grid):\n")
print(df_angle.to_string(index=False))

Performance by Condition x Angle (12 x 4 grid):

   Condition Angle   N     L2     DX Acc_L2 Acc_DX
   Stimuli_A  left 303 0.1182 0.0669  50.8%  46.2%
   Stimuli_A front 306 0.2446 0.2192  59.8%  59.5%
   Stimuli_A right 298 0.1180 0.0714  53.7%  55.0%
     Natural  left 111 0.1981 0.1587  48.6%  46.8%
     Natural front 111 0.1659 0.1231  64.0%  64.9%
     Natural right 111 0.1473 0.1153  58.6%  56.8%
  Consistent  left 111 0.1832 0.1426  51.4%  50.5%
  Consistent front 110 0.1601 0.1137  67.3%  68.2%
  Consistent right 111 0.1374 0.1017  62.2%  60.4%
Inconsistent  left 234 0.2781 0.2432  25.6%  25.2%
Inconsistent front 233 0.3916 0.3668  20.2%  19.7%
Inconsistent right 234 0.2968 0.2651  24.8%  25.6%


In [17]:
# Analysis for Inconsistent condition: Moondream's predictions
# Under each distance metric (L2 and DX), how often does it pick head_target, label, or neither?
# Split by n_candidates (2, 3, 4)

n_candidates_vals = [2, 3, 4]
metrics_names = ["L2", "DX"]

# Initialize counters
inconsistent_analysis = {}
for n_cand in n_candidates_vals:
    for metric in metrics_names:
        inconsistent_analysis[(n_cand, metric)] = {
            "head_target": 0,
            "label": 0,
            "neither": 0,
            "total": 0
        }

# Add aggregated counters
for metric in metrics_names:
    inconsistent_analysis[("all", metric)] = {
        "head_target": 0,
        "label": 0,
        "neither": 0,
        "total": 0
    }

for entry in dataset:
    stim_id = str(entry["stimulus_id"])
    if stim_id not in results:
        continue
    
    result = results[stim_id]
    if result["gaze"] is None:
        continue
    
    # Only inconsistent condition
    condition = get_condition(entry)
    if condition != "Inconsistent":
        continue
    
    pred_x, pred_y = result["gaze"]["x"], result["gaze"]["y"]
    
    options = get_options(entry)
    if len(options) < 2:
        continue
    
    n_cand = entry["n_candidates"]
    if n_cand not in n_candidates_vals:
        continue
    
    label = entry["label"]
    head_target = entry["head_target"]
    
    # Get predictions under both metrics
    nearest_l2 = min(options, key=lambda o: np.sqrt((pred_x - o["x"])**2 + (pred_y - o["y"])**2))
    nearest_dx = min(options, key=lambda o: np.abs(pred_x - o["x"]))
    
    # Categorize L2 prediction
    key_l2 = (n_cand, "L2")
    inconsistent_analysis[key_l2]["total"] += 1
    if nearest_l2["name"] == head_target:
        inconsistent_analysis[key_l2]["head_target"] += 1
    elif nearest_l2["name"] == label:
        inconsistent_analysis[key_l2]["label"] += 1
    else:
        inconsistent_analysis[key_l2]["neither"] += 1
    
    # Also add to aggregated
    key_l2_agg = ("all", "L2")
    inconsistent_analysis[key_l2_agg]["total"] += 1
    if nearest_l2["name"] == head_target:
        inconsistent_analysis[key_l2_agg]["head_target"] += 1
    elif nearest_l2["name"] == label:
        inconsistent_analysis[key_l2_agg]["label"] += 1
    else:
        inconsistent_analysis[key_l2_agg]["neither"] += 1
    
    # Categorize DX prediction
    key_dx = (n_cand, "DX")
    inconsistent_analysis[key_dx]["total"] += 1
    if nearest_dx["name"] == head_target:
        inconsistent_analysis[key_dx]["head_target"] += 1
    elif nearest_dx["name"] == label:
        inconsistent_analysis[key_dx]["label"] += 1
    else:
        inconsistent_analysis[key_dx]["neither"] += 1
    
    # Also add to aggregated
    key_dx_agg = ("all", "DX")
    inconsistent_analysis[key_dx_agg]["total"] += 1
    if nearest_dx["name"] == head_target:
        inconsistent_analysis[key_dx_agg]["head_target"] += 1
    elif nearest_dx["name"] == label:
        inconsistent_analysis[key_dx_agg]["label"] += 1
    else:
        inconsistent_analysis[key_dx_agg]["neither"] += 1

# Build analysis table
print("Inconsistent Condition: Prediction Distribution (Head Target vs Label vs Neither)\n")

analysis_data = []
for n_cand in n_candidates_vals:
    for metric in metrics_names:
        key = (n_cand, metric)
        counts = inconsistent_analysis[key]
        total = counts["total"]
        if total == 0:
            continue
        
        analysis_data.append({
            "n_candidates": n_cand,
            "Metric": metric,
            "N": total,
            "Head_Target": f"{counts['head_target']} ({counts['head_target']/total*100:.1f}%)",
            "Label": f"{counts['label']} ({counts['label']/total*100:.1f}%)",
            "Neither": f"{counts['neither']} ({counts['neither']/total*100:.1f}%)"
        })

# Add aggregated rows
for metric in metrics_names:
    key = ("all", metric)
    counts = inconsistent_analysis[key]
    total = counts["total"]
    if total > 0:
        analysis_data.append({
            "n_candidates": "All (agg)",
            "Metric": metric,
            "N": total,
            "Head_Target": f"{counts['head_target']} ({counts['head_target']/total*100:.1f}%)",
            "Label": f"{counts['label']} ({counts['label']/total*100:.1f}%)",
            "Neither": f"{counts['neither']} ({counts['neither']/total*100:.1f}%)"
        })

df_inconsistent = pd.DataFrame(analysis_data)
print(df_inconsistent.to_string(index=False))

Inconsistent Condition: Prediction Distribution (Head Target vs Label vs Neither)

n_candidates Metric   N Head_Target       Label     Neither
           2     L2 108  74 (68.5%)  34 (31.5%)    0 (0.0%)
           2     DX 108  74 (68.5%)  34 (31.5%)    0 (0.0%)
           3     L2 162  81 (50.0%)  40 (24.7%)  41 (25.3%)
           3     DX 162  79 (48.8%)  41 (25.3%)  42 (25.9%)
           4     L2 431 183 (42.5%)  91 (21.1%) 157 (36.4%)
           4     DX 431 179 (41.5%)  90 (20.9%) 162 (37.6%)
   All (agg)     L2 701 338 (48.2%) 165 (23.5%) 198 (28.2%)
   All (agg)     DX 701 332 (47.4%) 165 (23.5%) 204 (29.1%)


In [18]:
# Random guessing baseline for Stimuli B (Site != "Site0")
# Baseline accuracy = 1 / n_candidates

def compute_random_baseline():
    """
    Compute random guessing baseline (1/n_candidates) for Stimuli B
    Returns baseline accuracy for each condition and n_candidates
    """
    # Count distribution of n_candidates for each Stimuli B condition
    condition_n_cand_counts = {}
    
    for entry in dataset:
        condition = get_condition(entry)
        # Only Stimuli B (Natural, Consistent, Inconsistent)
        if condition == "Stimuli_A":
            continue
        
        n_cand = entry["n_candidates"]
        key = (condition, n_cand)
        
        if key not in condition_n_cand_counts:
            condition_n_cand_counts[key] = 0
        condition_n_cand_counts[key] += 1
    
    # Compute weighted baseline for each condition
    baseline_data = []
    
    for condition in ["Natural", "Consistent", "Inconsistent"]:
        total_samples = 0
        weighted_sum = 0
        
        for n_cand in [2, 3, 4]:
            key = (condition, n_cand)
            if key in condition_n_cand_counts:
                count = condition_n_cand_counts[key]
                total_samples += count
                weighted_sum += count * (1.0 / n_cand)
        
        if total_samples > 0:
            baseline_acc = weighted_sum / total_samples
            baseline_data.append({
                "Condition": condition,
                "N": total_samples,
                "Random_Baseline": f"{baseline_acc*100:.1f}%"
            })
    
    # Compute aggregated baseline for all Stimuli B
    total_stimuli_b = 0
    weighted_sum_b = 0
    for condition in ["Natural", "Consistent", "Inconsistent"]:
        for n_cand in [2, 3, 4]:
            key = (condition, n_cand)
            if key in condition_n_cand_counts:
                count = condition_n_cand_counts[key]
                total_stimuli_b += count
                weighted_sum_b += count * (1.0 / n_cand)
    
    if total_stimuli_b > 0:
        baseline_data.append({
            "Condition": "Stimuli_B (agg)",
            "N": total_stimuli_b,
            "Random_Baseline": f"{weighted_sum_b/total_stimuli_b*100:.1f}%"
        })
    
    # Also compute by n_candidates across all Stimuli B
    print("Random Guessing Baseline for Stimuli B:\n")
    print("By Condition:")
    df_baseline = pd.DataFrame(baseline_data)
    print(df_baseline.to_string(index=False))
    
    print("\n\nBy n_candidates (across all Stimuli B):")
    n_cand_data = []
    for n_cand in [2, 3, 4]:
        count = sum(condition_n_cand_counts.get((c, n_cand), 0) 
                   for c in ["Natural", "Consistent", "Inconsistent"])
        if count > 0:
            n_cand_data.append({
                "n_candidates": n_cand,
                "N": count,
                "Random_Baseline": f"{100.0/n_cand:.1f}%"
            })
    df_n_cand = pd.DataFrame(n_cand_data)
    print(df_n_cand.to_string(index=False))

compute_random_baseline()

Random Guessing Baseline for Stimuli B:

By Condition:
      Condition    N Random_Baseline
        Natural  333           35.1%
     Consistent  332           35.2%
   Inconsistent  701           30.8%
Stimuli_B (agg) 1366           32.9%


By n_candidates (across all Stimuli B):
 n_candidates   N Random_Baseline
            2 324           50.0%
            3 324           33.3%
            4 718           25.0%
